# Notebook 12: Recall@1000 Error Analysis

**Goal:** Understand WHY exact MiniLM search caps at Recall@1000 = 0.741 (seen in `03_baseline_minilm` and confirmed as the ANN ceiling in `11_ann_scaling_evaluation`). ANN hyperparameters cannot push past this number — they only approximate exact search. This notebook investigates whether the gap is a genuine embedding/semantic failure, or a data-quality artifact (boilerplate-duplicated company summaries, near-miss ranking noise, or noisy production ground truth).

No re-encoding needed — reuses the cached embeddings + FAISS index from `03_baseline_minilm`.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import faiss

RESULT_DIR = Path('result/12_recall_error_analysis')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/ — ready')

## 1. Load ground truth, queries, corpus, boilerplate audit

In [ ]:
print('[Load] Loading production ground truth...')
production_df = pd.read_excel('dataset/production_results.xlsx')
print(f'[Load] Production rows : {len(production_df):,}')

print('[Load] Loading queries...')
with open('dataset/goi_search_results.json', 'r') as f:
    queries_data = json.load(f)
print(f'[Load] Queries : {len(queries_data)}')

print('[Load] Loading corpus (for domain -> name/summary lookup)...')
all_companies = production_df.drop_duplicates(subset='domain').reset_index(drop=True)
domain_to_row = {row['domain']: row for _, row in all_companies.iterrows()}
print(f'[Load] Unique companies : {len(all_companies):,}')

print('[Load] Loading boilerplate-duplication audit (from trust_feature.py)...')
audit_csv = Path('result/summary_quality/summary_quality_audit.csv')
if audit_csv.exists():
    audit_df = pd.read_csv(audit_csv)
    dup_map = dict(zip(audit_df['domain'], audit_df['duplicate_count']))
    print(f'[Load] Flagged (duplicate_count>=5) domains : {len(dup_map):,}')
else:
    dup_map = {}
    print('[Load] No audit CSV found -- treating all domains as unique')

## 2. Load cached MiniLM embeddings + exact FAISS index

In [ ]:
EMB_PATH   = Path('result/03_baseline_minilm/company_embeddings.npy')
INDEX_PATH = Path('result/03_baseline_minilm/company_faiss.index')

print('[FAISS] Loading cached embeddings...')
embeddings = np.load(EMB_PATH).astype('float32')
print(f'[FAISS] Embeddings shape : {embeddings.shape}')

print('[FAISS] Loading cached exact index...')
index = faiss.read_index(str(INDEX_PATH))
print(f'[FAISS] Vectors in index : {index.ntotal:,}')

domain_by_idx = all_companies['domain'].tolist()
assert len(domain_by_idx) == index.ntotal, 'Corpus/index size mismatch'

## 3. Extended-depth retrieval (k=10,000)

Searching much deeper than k=1000 lets us tell apart:
- **near misses** -- the true company was ranked just outside the top-1000 (recoverable by widening k)
- **far / not-found misses** -- the true company is nowhere close, a genuine semantic failure of the embedding

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[Model] Loading MiniLM (all-MiniLM-L6-v2) on {DEVICE}...')
t0 = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print(f'[Model] Loaded in {time.time()-t0:.1f}s')

EXTENDED_K = 10_000
print(f'[Retrieval] Running extended search (k={EXTENDED_K}) for {len(queries_data)} queries...')

retrieved_ranks = {}   # query_id -> {domain: rank (1-indexed)}
t0 = time.time()
for i, item in enumerate(queries_data):
    qid   = item['query_id']
    query = item['query']
    query_emb = model.encode([query], normalize_embeddings=False, convert_to_numpy=True).astype('float32')
    _, idxs = index.search(query_emb, EXTENDED_K)
    retrieved_ranks[qid] = {domain_by_idx[idx]: rank + 1 for rank, idx in enumerate(idxs[0])}
    if (i + 1) % 20 == 0 or (i + 1) == len(queries_data):
        print(f'[Retrieval] {i+1}/{len(queries_data)} queries done')
print(f'[Retrieval] Done in {time.time()-t0:.1f}s')

## 4. Identify misses at k=1000 and classify them

In [ ]:
K_CUTOFF = 1000
miss_rows = []

for item in queries_data:
    qid   = item['query_id']
    query = item['query']
    relevant = set(production_df[
        (production_df['query_id'] == qid) &
        (production_df['rank'] <= K_CUTOFF)
    ]['domain'].tolist())

    ranks_for_query = retrieved_ranks[qid]

    for domain in relevant:
        retrieved_rank = ranks_for_query.get(domain)  # None if not found even at k=10,000
        if retrieved_rank is not None and retrieved_rank <= K_CUTOFF:
            continue  # correctly retrieved, not a miss

        row = domain_to_row.get(domain, {})
        prod_rank_row = production_df[
            (production_df['query_id'] == qid) & (production_df['domain'] == domain)
        ]
        prod_rank = int(prod_rank_row['rank'].iloc[0]) if len(prod_rank_row) else None

        if retrieved_rank is None:
            bucket = f'not_found_beyond_{EXTENDED_K}'
        elif retrieved_rank <= 2000:
            bucket = 'near_miss_1000_2000'
        elif retrieved_rank <= 5000:
            bucket = 'mid_miss_2000_5000'
        else:
            bucket = 'far_miss_5000_10000'

        miss_rows.append({
            'query_id':        qid,
            'query':           query,
            'domain':          domain,
            'name':            row.get('name', ''),
            'summary':         row.get('summary', ''),
            'production_rank': prod_rank,
            'retrieved_rank':  retrieved_rank,
            'bucket':          bucket,
            'duplicate_count': dup_map.get(domain, 1),
            'is_boilerplate':  dup_map.get(domain, 1) >= 5,
        })

miss_df = pd.DataFrame(miss_rows)
miss_df.to_csv(RESULT_DIR / 'recall_1000_misses.csv', index=False)
print(f'[Miss] Total missed (query, domain) pairs : {len(miss_df):,}')
print(f'[Miss] Saved to {RESULT_DIR}/recall_1000_misses.csv')

## 5. Baseline boilerplate rate (for comparison)

In [ ]:
all_relevant_rows = []
for item in queries_data:
    qid = item['query_id']
    relevant = production_df[(production_df['query_id'] == qid) & (production_df['rank'] <= K_CUTOFF)]
    for domain in relevant['domain'].tolist():
        all_relevant_rows.append(domain)

total_relevant_labels = len(all_relevant_rows)
baseline_boilerplate_rate = np.mean([dup_map.get(d, 1) >= 5 for d in all_relevant_rows])

print(f'[Baseline] Total ground-truth relevant labels (all queries, k<=1000) : {total_relevant_labels:,}')
print(f'[Baseline] Boilerplate rate among ALL relevant labels    : {baseline_boilerplate_rate:.1%}')
print(f'[Baseline] Boilerplate rate among MISSED labels          : {miss_df["is_boilerplate"].mean():.1%}')

## 6. Aggregate breakdown

In [ ]:
print('=' * 60)
print('MISS BREAKDOWN BY BUCKET')
print('=' * 60)
bucket_counts = miss_df['bucket'].value_counts()
for bucket, count in bucket_counts.items():
    pct = count / len(miss_df) * 100
    print(f'  {bucket:<28} {count:>6,}  ({pct:5.1f}%)')

print()
print('=' * 60)
print('MISS BREAKDOWN BY PRODUCTION-RANK CONFIDENCE')
print('=' * 60)
rank_buckets = [(1, 10, 'rank 1-10 (high confidence)'),
                (11, 50, 'rank 11-50'),
                (51, 200, 'rank 51-200'),
                (201, 500, 'rank 201-500'),
                (501, 1000, 'rank 501-1000 (low confidence)')]
for lo, hi, label in rank_buckets:
    sub = miss_df[(miss_df['production_rank'] >= lo) & (miss_df['production_rank'] <= hi)]
    print(f'  {label:<32} {len(sub):>6,} misses')

print()
print('=' * 60)
print('SUMMARY')
print('=' * 60)
near_recoverable = bucket_counts.get('near_miss_1000_2000', 0) + bucket_counts.get('mid_miss_2000_5000', 0)
print(f'  Total misses                                : {len(miss_df):,}')
print(f'  Recoverable by widening k to 5,000          : {near_recoverable:,} ({near_recoverable/len(miss_df)*100:.1f}%)')
print(f'  Genuinely far / not found within k=10,000   : {bucket_counts.get("far_miss_5000_10000", 0) + bucket_counts.get(f"not_found_beyond_{EXTENDED_K}", 0):,}')
print(f'  Flagged as boilerplate-duplicated summary   : {miss_df["is_boilerplate"].sum():,} ({miss_df["is_boilerplate"].mean()*100:.1f}%)')

## 7. Cross-reference with existing LLM relevance judgments

`result/08_llm_relevance_judge/llm_judgements.json` has LLM-judged relevance (score 0/1/2) for the top-20 production results on 20 held-out queries. This only covers shallow production ranks, so overlap with misses (which skew deep) will likely be small -- but any overlap tells us whether production's own ground truth is trustworthy at the ranks where misses occur.

In [ ]:
llm_judge_path = Path('result/08_llm_relevance_judge/llm_judgements.json')
if llm_judge_path.exists():
    llm_judgements = json.load(open(llm_judge_path))
    llm_lookup = {(j['query_id'], j['domain']): j['score'] for j in llm_judgements}

    overlap_rows = []
    for _, row in miss_df.iterrows():
        key = (row['query_id'], row['domain'])
        if key in llm_lookup:
            overlap_rows.append({**row.to_dict(), 'llm_score': llm_lookup[key]})

    print(f'[LLM-check] Misses overlapping with LLM-judged sample : {len(overlap_rows)} / {len(miss_df)}')
    if overlap_rows:
        overlap_df = pd.DataFrame(overlap_rows)
        print(overlap_df[['query', 'domain', 'production_rank', 'retrieved_rank', 'llm_score']].to_string(index=False))
        overlap_df.to_csv(RESULT_DIR / 'misses_with_llm_judgment.csv', index=False)
else:
    print('[LLM-check] No LLM judgment file found -- skipping')

## 8. Sample worst-case misses for manual inspection

The most informative cases: ground truth says this company is highly relevant (production rank <= 50) but MiniLM ranked it far outside the top-1000 (or didn't find it within k=10,000 at all).

In [ ]:
worst_misses = miss_df[miss_df['production_rank'] <= 50].sort_values(
    'retrieved_rank', ascending=False, na_position='first'
).head(15)

print('=' * 80)
print('WORST MISSES: production rank <= 50 but retrieved rank is far / not found')
print('=' * 80)
for _, row in worst_misses.iterrows():
    print(f"\nQuery              : {row['query']}")
    print(f"Missed company     : {row['name']}  ({row['domain']})")
    print(f"Production rank    : {row['production_rank']}   |   Retrieved rank : {row['retrieved_rank']}")
    print(f"Boilerplate flag   : {row['is_boilerplate']}  (duplicate_count={row['duplicate_count']})")
    print(f"Summary            : {str(row['summary'])[:300]}")

worst_misses.to_csv(RESULT_DIR / 'worst_misses_sample.csv', index=False)
print(f"\n[Saved] {RESULT_DIR}/worst_misses_sample.csv")

## 9. Findings

In [ ]:
print('[Findings] ' + '=' * 60)
print('[Findings] RECALL@1000 ERROR ANALYSIS -- SUMMARY')
print('[Findings] ' + '=' * 60)
print(f'''
  Total ground-truth relevant labels (all queries) : {total_relevant_labels:,}
  Total misses at k=1000                           : {len(miss_df):,}  (observed miss rate matches ~1 - 0.741 recall)

  Boilerplate-duplicated summary rate:
    Among ALL relevant labels   : {baseline_boilerplate_rate:.1%}
    Among MISSED labels         : {miss_df["is_boilerplate"].mean():.1%}
    (if the missed rate is much higher than baseline, duplicate/boilerplate summaries
     are a real, fixable contributor to the recall ceiling -- MiniLM cannot distinguish
     between companies that share an identical summary string)

  Recoverable by widening k to 5,000                : {near_recoverable:,} ({near_recoverable/len(miss_df)*100:.1f}% of misses)
  Genuinely far / not found within k=10,000          : {len(miss_df) - near_recoverable:,} ({(len(miss_df)-near_recoverable)/len(miss_df)*100:.1f}% of misses)

  These far misses are candidates for genuine embedding-quality failures (vocabulary
  mismatch, weak signal in the summary text) -- see worst_misses_sample.csv for concrete
  examples to inspect by hand.
''')
print(f'[Findings] All output files saved to {RESULT_DIR}/')